In [0]:
client_id     = dbutils.secrets.get(scope="keyvault3145", key="sp-client-id")
client_secret = dbutils.secrets.get(scope="keyvault3145", key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope="keyvault3145", key="sp-tenant-id")

In [0]:
# Config for Data Access from Azure Data Lake
spark.conf.set(
    "fs.azure.account.auth.type.mylakeplace31.dfs.core.windows.net", "OAuth")
spark.conf.set(
    "fs.azure.account.oauth.provider.type.mylakeplace31.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(
    "fs.azure.account.oauth2.client.id.mylakeplace31.dfs.core.windows.net",
    client_id)
spark.conf.set(
    "fs.azure.account.oauth2.client.secret.mylakeplace31.dfs.core.windows.net",
    client_secret)
spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.mylakeplace31.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
#Importing Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Creating parameter
YEAR= dbutils.widgets.get("YEAR")

In [0]:
print(YEAR)

2025


#### Create Database

In [0]:
# Create database if it doesn't exist
spark.sql("""
    CREATE DATABASE IF NOT EXISTS silver
    LOCATION 'abfss://silver@mylakeplace31.dfs.core.windows.net/'
""")

DataFrame[]

#### Trip type

In [0]:
# Creating triptype table
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.triptype
    USING DELTA
    LOCATION 'abfss://silver@mylakeplace31.dfs.core.windows.net/triptype'
""")

DataFrame[]

In [0]:
try:
    spark.read.table("silver.triptype")
    print("Table already exists, skipping load")

except:
    triptype = spark.read.format("csv") \
        .option("inferSchema", "true") \
        .option("header", "true") \
        .load("abfss://bronze@mylakeplace31.dfs.core.windows.net/trip_type/")

    triptype.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.triptype")

    print("Table created and data loaded")

Table already exists, skipping load


#### Trip zone

In [0]:
# Creating tripzone table
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.tripzone
    LOCATION 'abfss://silver@mylakeplace31.dfs.core.windows.net/tripzone'
""")

DataFrame[]

In [0]:
try:
    spark.read.table("silver.tripzone")
    print("Table already exists, skipping load")

except:
    tripzone = spark.read.format("csv")\
    .option("inferSchema","true")\
        .option("header","true")\
            .load("abfss://bronze@mylakeplace31.dfs.core.windows.net/trip_zone/")

    tripzone = tripzone.withColumn("Zone1",split(col("Zone"),'/')[0])\
    .withColumn("Zone2",expr("get(split(Zone, '/'), 1)"))\
    .drop("Zone")

    tripzone.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("silver.tripzone")

    print("Table created and data loaded")



Table already exists, skipping load


#### Trip Data

In [0]:
from pyspark.sql import functions as F

# CONFIG
STORAGE_ACCOUNT = "mylakeplace31"

SOURCE_PATH = f"abfss://bronze@mylakeplace31.dfs.core.windows.net/trip-data/{YEAR}/"
SILVER_TABLE = "silver.tripdata"
PROCESSED_FILES_PATH = f"abfss://silver@mylakeplace31.dfs.core.windows.net/metadata/processed_files"

# GET FILES
files = dbutils.fs.ls(SOURCE_PATH)
try:
    processed_df = spark.read.format("delta").load(PROCESSED_FILES_PATH)
    processed_files = [row.file_name for row in processed_df.collect()]
except:
    processed_files = []

In [0]:
new_files = [f.path for f in files if f.path not in processed_files]
print(f"Total new files to process: {len(new_files)}")

Total new files to process: 0


In [0]:
# PROCESS FILES
for file in new_files:

    try:
        print(f"\nProcessing file: {file}")

        df = spark.read.format("parquet").load(file)

        if "cbd_congestion_fee" not in df.columns:
            df = df.withColumn("cbd_congestion_fee", F.lit(None))

        # TRANSFORM
    
        df = (
            df
            .withColumnRenamed("VendorID", "vendor_id")
            .withColumnRenamed("RatecodeID", "ratecode_id")
            .withColumnRenamed("PULocationID", "pickup_location_id")
            .withColumnRenamed("DOLocationID", "dropoff_location_id")
            .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
            .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
        )

        df = (
            df
            .withColumn("vendor_id", F.col("vendor_id").cast("int"))
            .withColumn("ratecode_id", F.col("ratecode_id").cast("int"))
            .withColumn("pickup_location_id", F.col("pickup_location_id").cast("int"))
            .withColumn("dropoff_location_id", F.col("dropoff_location_id").cast("int"))
            .withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
            .withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))
            .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast("string"))
            .withColumn("passenger_count", F.col("passenger_count").cast("int"))
            .withColumn("trip_distance", F.col("trip_distance").cast("double"))
            .withColumn("fare_amount", F.col("fare_amount").cast("double"))
            .withColumn("extra", F.col("extra").cast("double"))
            .withColumn("mta_tax", F.col("mta_tax").cast("double"))
            .withColumn("tip_amount", F.col("tip_amount").cast("double"))
            .withColumn("tolls_amount", F.col("tolls_amount").cast("double"))
            .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast("double"))
            .withColumn("total_amount", F.col("total_amount").cast("double"))
            .withColumn("payment_type", F.col("payment_type").cast("int"))
            .withColumn("trip_type", F.col("trip_type").cast("int"))
            .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
            .withColumn("cbd_congestion_fee", F.col("cbd_congestion_fee").cast("double"))
        )

        df = df.drop("ehail_fee")

        # DERIVED
        df = (
            df
            .withColumn("trip_year", F.year("pickup_datetime"))
            .withColumn("trip_month", F.month("pickup_datetime"))
        )

        # FILTER
        df = df.filter(
        (F.year("pickup_datetime") == int(YEAR)) &
        (F.col("total_amount") > 0) &
        (F.col("trip_distance") > 0) &
        (F.col("passenger_count") > 0)
   )

        # COUNT (for logging)
        row_count = df.count()

        # WRITE
        df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .partitionBy("trip_year", "trip_month") \
            .saveAsTable(SILVER_TABLE)

        # TRACK FILE
        spark.createDataFrame([(file,)], ["file_name"]) \
            .write \
            .format("delta") \
            .mode("append") \
            .save(PROCESSED_FILES_PATH)

        # SUCCESS LOG
        print(f"Successfully processed: {file}")
        print(f"Rows written: {row_count}")

    except Exception as e:
        print(f"Error processing file: {file}")
        print(f"Error message: {str(e)}")